In [3]:
import os
import numpy as np
import matplotlib.pyplot as plt

def confidence_ellipse(ax, Sigma, n_std=2.0):
    """
    Plot an n-std confidence ellipse for a 2D Gaussian with covariance Sigma,
    centered at the origin. Uses eigen-decomposition to get ellipse axes/orientation.
    """
    # Eigen-decomposition (Sigma is symmetric PSD)
    vals, vecs = np.linalg.eigh(Sigma)
    order = np.argsort(vals)[::-1]
    vals, vecs = vals[order], vecs[:, order]

    # Semi-axis lengths
    a, b = n_std * np.sqrt(vals[0]), n_std * np.sqrt(vals[1])

    # Rotation angle from eigenvector of largest eigenvalue
    angle = np.arctan2(vecs[1, 0], vecs[0, 0])

    # Parametric ellipse
    t = np.linspace(0, 2*np.pi, 400)
    ellipse = np.vstack((a * np.cos(t), b * np.sin(t)))

    # Rotate ellipse
    R = np.array([[np.cos(angle), -np.sin(angle)],
                  [np.sin(angle),  np.cos(angle)]])
    ellipse_rot = R @ ellipse

    ax.plot(ellipse_rot[0, :], ellipse_rot[1, :], linewidth=1.5)  # default color


def make_case_plot(Sigma, title, outpath, n_samples=600, seed=0, lim=7.0):
    rng = np.random.default_rng(seed)
    mu = np.array([0.0, 0.0])

    # Sample points
    X = rng.multivariate_normal(mean=mu, cov=Sigma, size=n_samples)

    # Fixed points (same across all cases)
    x_a = np.array([0.0, 0.0])
    x_b = np.array([2.0, 0.0])

    fig, ax = plt.subplots(figsize=(5.2, 4.6))

    # Scatter cloud
    ax.scatter(X[:, 0], X[:, 1], s=10, alpha=0.6)

    # Confidence ellipse (helps show orientation/shape)
    confidence_ellipse(ax, Sigma, n_std=2.0)

    # Highlight points with distinct markers (no manual colors)
    ax.scatter([x_a[0]], [x_a[1]], s=160, marker="o", edgecolors="black", linewidths=1.5, label=r"$\mathbf{x}_a=(0,0)$")
    ax.scatter([x_b[0]], [x_b[1]], s=180, marker="*", edgecolors="black", linewidths=1.0, label=r"$\mathbf{x}_b=(2,0)$")

    ax.set_title(title)
    ax.set_xlabel(r"$x_1$")
    ax.set_ylabel(r"$x_2$")
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.grid(True, alpha=0.25)
    ax.legend(loc="upper right", frameon=True)

    os.makedirs(os.path.dirname(outpath), exist_ok=True)
    fig.tight_layout()
    fig.savefig(outpath, dpi=300)
    plt.close(fig)


def main():
    # Output directory structure matching your LaTeX paths
    out_dir = os.path.join("img", "lecture10")

    # Covariance matrices (exactly as in your notes)
    Sigma1 = np.array([[1.0,   0.0],
                       [0.0,   1.0]])

    Sigma2 = np.array([[4.0,   0.0],
                       [0.0,   1.0]])

    Sigma3 = np.array([[0.25,  0.0],
                       [0.0,   1.0]])

    Sigma4 = np.array([[1.0,   0.8],
                       [0.8,   1.0]])

    Sigma5 = np.array([[2.125, 1.875],
                       [1.875, 2.125]])

    cases = [
        (Sigma1, r"Case 1: Isotropic Gaussian ($\Sigma_1$)", os.path.join(out_dir, "case1_isotropic.png"), 101),
        (Sigma2, r"Case 2: Large variance in $x$ ($\Sigma_2$)", os.path.join(out_dir, "case2_large_var_x.png"), 102),
        (Sigma3, r"Case 3: Small variance in $x$ ($\Sigma_3$)", os.path.join(out_dir, "case3_small_var_x.png"), 103),
        (Sigma4, r"Case 4: Correlated features ($\Sigma_4$)", os.path.join(out_dir, "case4_correlated.png"), 104),
        (Sigma5, r"Case 5: Rotated ellipse ($\Sigma_5$)", os.path.join(out_dir, "case5_rotated.png"), 105),
    ]

    for Sigma, title, outpath, seed in cases:
        make_case_plot(
            Sigma=Sigma,
            title=title,
            outpath=outpath,
            n_samples=700,     # tweak if you want denser clouds
            seed=seed,         # deterministic per-case
            lim=7.0            # consistent axes across cases
        )

    print("Saved figures to:")
    for _, _, outpath, _ in cases:
        print("  -", outpath)


if __name__ == "__main__":
    main()

Saved figures to:
  - img/lecture10/case1_isotropic.png
  - img/lecture10/case2_large_var_x.png
  - img/lecture10/case3_small_var_x.png
  - img/lecture10/case4_correlated.png
  - img/lecture10/case5_rotated.png


## Interactive Controls\nUse the widget panel below to explore parameters live inside JupyterLite.\n

In [ ]:
# INTERACTIVE_WIDGET_SECTION\nimport numpy as np\nimport matplotlib.pyplot as plt\nimport ipywidgets as widgets\n\ndef _plot_mahalanobis(var_x=2.0,var_y=1.0,corr=0.0):\n    cov=np.array([[var_x, corr*np.sqrt(var_x*var_y)],[corr*np.sqrt(var_x*var_y), var_y]])\n    cov += 1e-6*np.eye(2)\n    vals, vecs = np.linalg.eigh(cov)\n    t=np.linspace(0,2*np.pi,300)\n    circle=np.vstack([np.cos(t),np.sin(t)])\n    ellipse=(vecs @ np.diag(np.sqrt(vals)) @ circle)\n    plt.figure(figsize=(6,6))\n    plt.plot(ellipse[0], ellipse[1], lw=2)\n    plt.axhline(0,c='k',lw=0.8); plt.axvline(0,c='k',lw=0.8)\n    plt.gca().set_aspect('equal', adjustable='box')\n    plt.grid(True); plt.title('Interactive Covariance Ellipse')\n    plt.show()\n\nwidgets.interact(_plot_mahalanobis,\n    var_x=widgets.FloatSlider(value=2,min=0.2,max=5,step=0.1),\n    var_y=widgets.FloatSlider(value=1,min=0.2,max=5,step=0.1),\n    corr=widgets.FloatSlider(value=0,min=-0.95,max=0.95,step=0.05))\n